### Notebook was created in Google Colab

In [1]:
import pandas as pd

df_pairs = pd.read_csv('../data/processed/matching_pairs.csv')
df_resume = pd.read_csv('../data/processed/resume_clean.csv')
df_jobs = pd.read_parquet('../data/processed/jobs_clean.parquet')


In [5]:
print(df_pairs.shape)
print(df_pairs.head(3))

(29796, 4)
                                         resume_text  \
0  hr administrator marketing associate hr admini...   
1  hr administrator marketing associate hr admini...   
2  hr administrator marketing associate hr admini...   

                                           job_title  \
0                         Human Resources Generalist   
1  TWIC On-Call Crew Transportation Driver - Corp...   
2  Sales Associate - The Cosmetics Company Store ...   

                                     job_description  label  
0  job description with supervision the hr genera...      1  
1  danner s inc a leading maritime transportation...      1  
2  position summary as one of our highly skilled ...      1  


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2', device="mps")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6658.96it/s]


In [7]:
df_pairs['job_title_description'] = df_pairs['job_title'].str.lower() + " " + df_pairs['job_description']


resume_embeddings = model.encode(df_pairs['resume_text'].to_list(), convert_to_tensor=True)


job_embeddings = model.encode(df_pairs['job_title_description'].to_list(), convert_to_tensor=True)

In [8]:
from sentence_transformers import util
cosine_scores = util.cos_sim(resume_embeddings, job_embeddings).diagonal()
print(cosine_scores.shape)
print(cosine_scores[:5])

torch.Size([29796])
tensor([0.4459, 0.3317, 0.4119, 0.2216, 0.4358], device='mps:0')


In [9]:
df_pairs['cosine_score'] = cosine_scores.cpu().numpy()

In [10]:
def skill_overlap(resume_text, job_title_description):
  resume_text = set(resume_text.split())
  job_title_description = set(job_title_description.split())

  common = job_title_description.intersection(resume_text)
  ratio = len(common) / len(job_title_description) if job_title_description else 0
  return ratio

df_pairs['skill_overlap'] = df_pairs.apply(
    lambda row: skill_overlap(row['resume_text'], row['job_title_description']), axis=1
)

df_pairs.head(3)

,resume_text,job_title,job_description,label,job_title_description,cosine_score,skill_overlap
0,hr administrator marketing associate hr admini...,Human Resources Generalist,job description with supervision the hr genera...,1,human resources generalist job description wit...,0.445917,0.201946
1,hr administrator marketing associate hr admini...,TWIC On-Call Crew Transportation Driver - Corp...,danner s inc a leading maritime transportation...,1,twic on-call crew transportation driver - corp...,0.331657,0.259615
2,hr administrator marketing associate hr admini...,Sales Associate - The Cosmetics Company Store ...,position summary as one of our highly skilled ...,1,sales associate - the cosmetics company store ...,0.411923,0.210884


In [11]:
print(df_pairs['skill_overlap'].describe())

count    29796.000000
mean         0.202273
std          0.066187
min          0.000000
25%          0.157530
50%          0.195122
75%          0.239451
max          0.588235
Name: skill_overlap, dtype: float64


In [12]:
def text_length_ratio(resume_text, job_title_description):
  if len(resume_text) > len(job_title_description):
    res = len(job_title_description) / len(resume_text)
  elif len(job_title_description) > len(resume_text):
    res = len(resume_text) / len(job_title_description)
  else:
    res = 1
  return res

In [13]:
df_pairs['text_lenght_ratio'] = df_pairs.apply(
    lambda row: text_length_ratio(row['resume_text'], row['job_title_description']), axis=1
)
df_pairs.head()

,resume_text,job_title,job_description,label,job_title_description,cosine_score,skill_overlap,text_lenght_ratio
0,hr administrator marketing associate hr admini...,Human Resources Generalist,job description with supervision the hr genera...,1,human resources generalist job description wit...,0.445917,0.201946,0.919221
1,hr administrator marketing associate hr admini...,TWIC On-Call Crew Transportation Driver - Corp...,danner s inc a leading maritime transportation...,1,twic on-call crew transportation driver - corp...,0.331657,0.259615,0.184606
2,hr administrator marketing associate hr admini...,Sales Associate - The Cosmetics Company Store ...,position summary as one of our highly skilled ...,1,sales associate - the cosmetics company store ...,0.411923,0.210884,0.749537
3,hr administrator marketing associate hr admini...,Quality Supervisor,location sumter sc about skf skf has been maki...,0,quality supervisor location sumter sc about sk...,0.221553,0.188679,0.928885
4,hr administrator marketing associate hr admini...,Corporate Director of Reimbursement (Exempt),facility cape fear valley medical center locat...,0,corporate director of reimbursement (exempt) f...,0.435810,0.278788,0.396995


In [14]:
X = df_pairs[['cosine_score', 'skill_overlap', 'text_lenght_ratio']]
y = df_pairs['label']

In [15]:
# from sklearn.model_selection import train_test_split
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     random_state=42,
#     test_size=0.3
# )

In [ ]:
# from sklearn.utils import class_weight
# import xgboost as xgb
#
# model = xgb.XGBClassifier(
#     n_estimators=100,
#     scale_pos_weight=3,
#     random_state=42
# )
#
# model.fit(X_train, y_train)

In [ ]:
# from sklearn.metrics import classification_report
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


resume_np = resume_embeddings.cpu().numpy()
job_np = job_embeddings.cpu().numpy()

diff = np.abs(resume_np - job_np)

X_emb = np.hstack([
    df_pairs[['cosine_score', 'skill_overlap', 'text_lenght_ratio']].values,
    diff
])

X_train, X_test, y_train, y_test = train_test_split(
    X_emb,
    y,
    random_state=42,
    test_size=0.3
)

In [ ]:
from sklearn.utils import class_weight
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=3,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

import joblib
joblib.dump(model, '...models/matching_model.joblib')

In [ ]:
import joblib
joblib.dump(model, '...models/matching_model.joblib')